# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")


## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# Examine available record sets and their @id
record_sets = dataset.metadata.record_sets
print("Available Record Sets:")
for rs in record_sets:
    print(f"- @id: {rs['@id']}, name: {rs.get('name', 'N/A')}")
    # List associated fields
    if 'fields' in rs:
        print("  Fields:")
        for field in rs['fields']:
            print(f"    @id: {field['@id']}, name: {field.get('name', 'N/A')}")

# Optionally preview some records from the first record set
if record_sets:
    record_set_id = record_sets[0]['@id']
    print(f"\nFirst records from record set (@id={record_set_id}):")
    for x in dataset.records(record_set=record_set_id):
        print(x)
        break  # Show only one example

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Collect all record set @ids
record_set_ids = [rs['@id'] for rs in record_sets]
dataframes = {}

for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    if records:
        dataframes[rs_id] = pd.DataFrame(records)

# Display available DataFrame columns for first record set
if record_set_ids:
    print(dataframes[record_set_ids[0]].columns.tolist())
    dataframes[record_set_ids[0]].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
from numpy import nan
import numpy as np

# Choose a numeric field and a group field from the columns (replace below with actual @id if possible)
df = dataframes[record_set_ids[0]]
print(f"Columns: {df.columns.tolist()}")

# For demo: look for one likely numeric column such as GAD-7 score
numeric_candidates = [col for col in df.columns if ('gad' in col.lower() or 'score' in col.lower() or 'phq' in col.lower() or 'psq' in col.lower())]
print(f"Numeric candidates: {numeric_candidates}")

if numeric_candidates:
    numeric_field = numeric_candidates[0]
else:
    numeric_field = df.select_dtypes(include=np.number).columns[0] if not df.select_dtypes(include=np.number).empty else df.columns[0]

threshold = 10  # Generic threshold; adjust depending on available values
filtered_df = df[df[numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > {threshold}:")
print(filtered_df.head())

# Normalize the numeric field
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"Normalized {numeric_field} for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"].head())

# Choose a group field (like gender, age group, marital status etc)
group_candidates = [col for col in df.columns if ('gender' in col.lower() or 'age' in col.lower() or 'marital' in col.lower() or 'education' in col.lower())]
if group_candidates:
    group_field = group_candidates[0]
    grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
    print(f"Grouped data by {group_field}:")
    print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
# Example: Visualize numeric_field distribution
import matplotlib.pyplot as plt

if numeric_field in df.columns:
    plt.figure(figsize=(8,5))
    df[numeric_field].hist(bins=20)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

# Example: Grouped mean plot by group_field
if group_candidates and numeric_field in df.columns:
    group_field = group_candidates[0]
    grouped = df.groupby(group_field)[numeric_field].mean()
    grouped.plot(kind='bar', figsize=(8,5))
    plt.title(f"Average {numeric_field} by {group_field}")
    plt.ylabel(f"Mean {numeric_field}")
    plt.xlabel(group_field)
    plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- This notebook demonstrated loading, overview, extraction, and analysis of the FAIR² Mental Health Survey Dataset from Kilifi County, Kenya using the `mlcroissant` library.
- The dataset contains detailed survey records, including demographic and mental health assessment scores from GAD-7, PHQ-9, and PSQ.
- After loading data via the record set and field `@id` references, key numeric fields were explored and visualized.
- Filtering, normalization, and grouping revealed insights into survey participant distributions and trends.

For further work, deeper analysis of specific fields, modeling, and cross-record set integration can be performed using the flexible Python API and Croissant schema.